In [ ]:
!pip install -q fastapi uvicorn pyngrok "diffusers[torch]" accelerate safetensors

from fastapi import FastAPI
from fastapi.responses import FileResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from pyngrok import ngrok
import torch, tempfile, os
from diffusers import StableDiffusionXLPipeline
import uvicorn, threading

app = FastAPI()

# Allow requests from your frontend (for testing, allow all)
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# ----- load model -----
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    use_safetensors=True,
).to(device)


class Txt2ImgRequest(BaseModel):
    prompt: str


@app.post("/api/text2img")
def text2img(req: Txt2ImgRequest):
    img = pipe(
        req.prompt,
        num_inference_steps=30,
        guidance_scale=7.5
    ).images[0]
    tmp = tempfile.mkdtemp()
    path = os.path.join(tmp, "out.png")
    img.save(path)
    return FileResponse(path, media_type="image/png")


def run():
    uvicorn.run(app, host="0.0.0.0", port=8000)


thread = threading.Thread(target=run, daemon=True)
thread.start()

# ---- ngrok ----
ngrok.set_auth_token("37MvwGSAtToQLoLlAYagYLtxMk0_5KNCdcjUwNQiufDMn8DEs")
public_url = ngrok.connect(8000, "http")
public_url


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

INFO:     Started server process [2651]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


<NgrokTunnel: "https://nondiffuse-unrespectable-lissa.ngrok-free.dev" -> "http://localhost:8000">